In [3]:
import math

import torch

batch_size = 1
seq_len = 3
d_model = 4

X = torch.tensor([
    [
        [1., 0., 1., 0.],
        [0., 2., 0., 2.],
        [1., 1., 1., 1.]
    ]
])

print(X.shape)

torch.Size([1, 3, 4])


In [4]:
import torch.nn as nn
Wq = nn.Linear(4, 4, bias=False)
Wk = nn.Linear(4, 4, bias=False)
Wv = nn.Linear(4, 4, bias=False)

In [5]:
Q = Wq(X)
K = Wk(X)
V = Wv(X)

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

Q shape: torch.Size([1, 3, 4])
K shape: torch.Size([1, 3, 4])
V shape: torch.Size([1, 3, 4])


In [6]:
num_heads = 2
head_dim = 2
def split_heads(x, num_heads, head_dim):
    batch, seq_len, _ = x.shape
    x = x.view(batch, seq_len, num_heads, head_dim)
    return x.transpose(1, 2)

Q_h = split_heads(Q, num_heads, head_dim)
K_h = split_heads(K, num_heads, head_dim)
V_h = split_heads(V, num_heads, head_dim)

print("Q_h shape:", Q_h.shape)
print("K_h shape:", K_h.shape)
print("V_h shape:", V_h.shape)


Q_h shape: torch.Size([1, 2, 3, 2])
K_h shape: torch.Size([1, 2, 3, 2])
V_h shape: torch.Size([1, 2, 3, 2])


In [7]:
scores = Q_h @ K_h.transpose(-2, -1)          # [1, 2, 3, 3]
scores = scores / math.sqrt(head_dim)
weights = torch.softmax(scores, dim=-1)        # [1, 2, 3, 3]
output = weights @ V_h                         # [1, 2, 3, 2]

print("scores shape:", scores.shape)
print("weights shape:", weights.shape)
print("output shape:", output.shape)

scores shape: torch.Size([1, 2, 3, 3])
weights shape: torch.Size([1, 2, 3, 3])
output shape: torch.Size([1, 2, 3, 2])


In [8]:
def combine_heads(x):
    batch, num_heads, seq_len, head_dim = x.shape
    x = x.transpose(1, 2).contiguous().view(batch, seq_len, num_heads * head_dim)
    return x

In [9]:
combined_output = combine_heads(output)
print(f"combined_output shape: {combined_output.shape}")
wo = nn.Linear(4, 4, bias=False)
final_output = wo(combined_output)
print(final_output.shape)

combined_output shape: torch.Size([1, 3, 4])
torch.Size([1, 3, 4])
